In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/football_data")

for f in sorted(RAW.glob("E0_*.csv")):
    df = pd.read_csv(f)
    print(f"{f.name}:  {df.shape[0]} rows, {df.shape[1]} columns")

E0_1415.csv:  381 rows, 68 columns
E0_1516.csv:  380 rows, 65 columns
E0_1617.csv:  380 rows, 65 columns
E0_1718.csv:  380 rows, 65 columns
E0_1819.csv:  380 rows, 62 columns
E0_1920.csv:  380 rows, 106 columns
E0_2021.csv:  380 rows, 106 columns
E0_2122.csv:  380 rows, 106 columns
E0_2223.csv:  380 rows, 106 columns
E0_2324.csv:  380 rows, 106 columns
E0_2425.csv:  380 rows, 120 columns
E0_2526.csv:  380 rows, 132 columns


In [2]:
df_1415 = pd.read_csv(RAW / "E0_1415.csv")
df_1415.tail(3)

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA
378,E0,24/05/15,Newcastle,West Ham,2.0,0.0,H,0.0,0.0,D,...,2.25,25.0,-0.50,1.82,1.78,2.20,2.10,1.76,4.01,4.98
379,E0,24/05/15,Stoke,Liverpool,6.0,1.0,H,5.0,0.0,H,...,1.99,25.0,0.25,2.07,2.02,1.88,1.85,3.56,3.60,2.17
380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print([c for c in df_1415.columns if c.startswith("Unnamed")])

[]


In [4]:
columns_by_season = {f.name: set(pd.read_csv(f).columns) for f in sorted(RAW.glob("E0_*.csv"))}

common = set.intersection(*columns_by_season.values())
print(f"{len(common)} columns appear in ALL 12 seasons:\n")
print(sorted(common))

35 columns appear in ALL 12 seasons:

['AC', 'AF', 'AR', 'AS', 'AST', 'AY', 'AwayTeam', 'B365A', 'B365D', 'B365H', 'BWA', 'BWD', 'BWH', 'Date', 'Div', 'FTAG', 'FTHG', 'FTR', 'HC', 'HF', 'HR', 'HS', 'HST', 'HTAG', 'HTHG', 'HTR', 'HY', 'HomeTeam', 'PSA', 'PSCA', 'PSCD', 'PSCH', 'PSD', 'PSH', 'Referee']


In [5]:
NEEDED = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
          "HS", "AS", "HST", "AST", "B365H", "B365D", "B365A"]

for season, cols in columns_by_season.items():
    missing = [c for c in NEEDED if c not in cols]
    print(f"{season}: {'✅ all present' if not missing else '❌ missing ' + str(missing)}")

E0_1415.csv: ✅ all present
E0_1516.csv: ✅ all present
E0_1617.csv: ✅ all present
E0_1718.csv: ✅ all present
E0_1819.csv: ✅ all present
E0_1920.csv: ✅ all present
E0_2021.csv: ✅ all present
E0_2122.csv: ✅ all present
E0_2223.csv: ✅ all present
E0_2324.csv: ✅ all present
E0_2425.csv: ✅ all present
E0_2526.csv: ✅ all present


In [6]:
# Rows where there's no home team = not a real match
bad_rows = df_1415[df_1415["HomeTeam"].isna()]
print(f"{len(bad_rows)} row(s) with no HomeTeam")
bad_rows

1 row(s) with no HomeTeam


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA
380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
dupes = df_1415[df_1415.duplicated(subset=["Date", "HomeTeam", "AwayTeam"], keep=False)]
print(f"{len(dupes)} duplicated row(s)")
dupes

0 duplicated row(s)


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA


In [8]:
for f in sorted(RAW.glob("E0_*.csv")):
    sample = pd.read_csv(f)["Date"].dropna().iloc[0]
    print(f"{f.name}: first date = {sample}")

E0_1415.csv: first date = 16/08/14
E0_1516.csv: first date = 08/08/2015
E0_1617.csv: first date = 13/08/16
E0_1718.csv: first date = 11/08/2017
E0_1819.csv: first date = 10/08/2018
E0_1920.csv: first date = 09/08/2019
E0_2021.csv: first date = 12/09/2020
E0_2122.csv: first date = 13/08/2021
E0_2223.csv: first date = 05/08/2022
E0_2324.csv: first date = 11/08/2023
E0_2425.csv: first date = 16/08/2024
E0_2526.csv: first date = 15/08/2025


In [9]:
m = pd.read_csv("../data/processed/matches.csv")

odds = m[["odds_home", "odds_draw", "odds_away"]]
m["margin"] = (1 / odds).sum(axis=1)

print(m["margin"].describe().round(4))
print("\nTop 5 highest margins:")
m.sort_values("margin", ascending=False)[["season", "date", "home_team", "away_team", "odds_home", "odds_draw", "odds_away", "margin"]].head()

count    4560.0000
mean        1.0435
std         0.0150
min         1.0169
25%         1.0274
50%         1.0499
75%         1.0553
max         1.1667
Name: margin, dtype: float64

Top 5 highest margins:


,season,date,home_team,away_team,odds_home,odds_draw,odds_away,margin
3261,2022/23,2023-02-18,Aston Villa,Arsenal,4.50,3.60,1.5,1.166667
4523,2025/26,2026-05-02,Wolves,Sunderland,3.10,3.20,2.1,1.111271
4380,2025/26,2026-01-06,West Ham,Nott'm Forest,2.88,3.25,2.2,1.109460
2599,2020/21,2021-04-22,Leicester,West Brom,1.53,3.80,6.0,1.083419
3084,2022/23,2022-08-31,Arsenal,Aston Villa,1.45,4.50,6.0,1.078544
